In [0]:
%python
import mlflow
from langchain_community.vectorstores import DatabricksVectorSearch
from databricks.vector_search.client import VectorSearchClient
from langchain_community.chat_models import ChatDatabricks
from langchain.prompts import PromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

# Configuração do MLflow no Unity Catalog
mlflow.set_registry_uri("databricks-uc")
MODEL_NAME = "sales_tax_catalog.gold.sales_tax_rag_bot"

def build_and_log_rag_chain():
    try:
        # 1. Configura o cliente de busca vetorial
        vsc = VectorSearchClient()
        index = vsc.get_index(endpoint_name="sales_tax_vector_endpoint", 
                              index_name="sales_tax_catalog.gold.sales_tax_index")
        
        vector_store = DatabricksVectorSearch(index, text_column="document_text")
        # Define a IA para recuperar os 5 contextos mais relevantes (k=5)
        retriever = vector_store.as_retriever(search_kwargs={"k": 5})

        # 2. Configura o LLM hospedado no Databricks
        llm = ChatDatabricks(endpoint="databricks-meta-llama-3-70b-instruct")

        # 3. Criação do Prompt de Comando
        template = """
        You are an AI assistant specialized in Sales Tax Allocation by City.
        Use the following pieces of retrieved context to answer the question.
        If the context does not contain the answer, say "I don't have information on this specific tax allocation."
        
        Context: {context}
        Question: {question}
        Answer:
        """
        prompt = PromptTemplate(template=template, input_variables=["context", "question"])

        # 4. Construção da RAG Chain usando LangChain Expression Language (LCEL)
        rag_chain = (
            {"context": retriever, "question": RunnablePassthrough()} 
            | prompt 
            | llm 
            | StrOutputParser()
        )

        # 5. Registo do Modelo no Unity Catalog via MLflow
        with mlflow.start_run() as run:
            mlflow.langchain.log_model(
                lc_model=rag_chain,
                artifact_path="rag_chain",
                registered_model_name=MODEL_NAME
            )
        print(f"Modelo RAG registado no Unity Catalog como: {MODEL_NAME}")
        
    except Exception as e:
        print("\n--- AVISO DE AMBIENTE ---")
        print("Este código está correto para ambientes Databricks Premium (Unity Catalog/MLflow).")
        print("A orquestração do LangChain requer acesso ao Vector Search e Model Serving,")
        print("que não estão habilitados na Free Edition.")
        print(f"Erro original: {e}")

if __name__ == "__main__":
    build_and_log_rag_chain()